In [1]:
import findspark

In [2]:
findspark.init("/opt/manual/spark")

In [3]:
from pyspark.sql import SparkSession, functions as F
import pandas as pd

In [4]:
spark = SparkSession.builder \
.appName("Churn Scoring with GBT") \
.master("yarn") \
.config("spark.sql.shuffle.partitions","2") \
.getOrCreate()

In [5]:
spark.version

'3.1.1'

In [7]:
! hdfs dfs -ls /user/train/datasets | grep Churn_Modelling

-rw-r--r--   1 train supergroup     674857 2022-01-16 12:20 /user/train/datasets/Churn_Modelling.csv


## Read train data

In [8]:
df = spark.read.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("hdfs://localhost:9000/user/train/datasets/Churn_Modelling.csv")

## Explore train data

In [9]:
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

In [10]:
df.limit(6).toPandas()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
5,6,15574012,Chu,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1


In [11]:
df.printSchema()

root
 |-- RowNumber: integer (nullable = true)
 |-- CustomerId: integer (nullable = true)
 |-- Surname: string (nullable = true)
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)



In [17]:
# Schema change analysis
# spark inference is consistent

In [12]:
df.persist()

DataFrame[RowNumber: int, CustomerId: int, Surname: string, CreditScore: int, Geography: string, Gender: string, Age: int, Tenure: int, Balance: double, NumOfProducts: int, HasCrCard: int, IsActiveMember: int, EstimatedSalary: double, Exited: int]

In [13]:
df_count = df.count()
print(df_count)

10000


In [14]:
len(df.columns)

14

# null check

In [15]:
# Null check
for col_name in df.dtypes:
    null_count = df.filter( (F.col(col_name[0]).isNull()) | (F.col(col_name[0]) == "")).count()
    if(  null_count > 0 ):
        print("{} {} type has {} null values % {}".format(col_name, col_type[1], null_count, (null_count/df_count * 100)))

In [16]:
# There is no null

# Group columns

In [17]:
categoric_cols = []
numeric_cols = []
discarted_cols = ['RowNumber','CustomerId','Surname']
label_col = ['Exited']

In [18]:
# column seperation
for col_name in df.dtypes:
    if (col_name[0] not in label_col+discarted_cols):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [19]:
print(numeric_cols)
print(len(numeric_cols))

['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
8


In [20]:
print(categoric_cols)
print(len(categoric_cols))

['Geography', 'Gender']
2


In [21]:
print(discarted_cols)
print(len(discarted_cols))

['RowNumber', 'CustomerId', 'Surname']
3


In [22]:
print(label_col)
print(len(label_col))

['Exited']
1


#  trim string columns

In [24]:
for col_name in categoric_cols:
    df = df.withColumn(col_name, F.trim(col_name))

# Distribution of Output/Target variable

In [27]:
df.select(label_col[0]).groupBy(label_col[0]).count().show()

+------+-----+
|Exited|count|
+------+-----+
|     1| 2037|
|     0| 7963|
+------+-----+



In [28]:
df.select(label_col[0]).groupBy(label_col[0]).count().toPandas().plot(kind='bar')

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

# Examine Categoricals

In [30]:
for col_name in categoric_cols:
    df.groupBy(col_name).agg(F.count("*").alias("TotalCount")).orderBy(F.desc("TotalCount")).show()

+---------+----------+
|Geography|TotalCount|
+---------+----------+
|   France|      5014|
|  Germany|      2509|
|    Spain|      2477|
+---------+----------+

+------+----------+
|Gender|TotalCount|
+------+----------+
|  Male|      5457|
|Female|      4543|
+------+----------+



## verification of columns

In [32]:
if len(df.columns) == (len(categoric_cols)+len(numeric_cols)+len(discarted_cols+label_col)):
    print("Columns verified")
else: print("Columns are not verified")

Columns verified


## Select columns for onehotencoder

In [33]:
# Here we count the distinct categories in categoric columns. If there is more than two,
# we will add those to to_be_onehotencoded_cols list If there is just 2 we don't need to use onehotencoder
# So if distinct category gt 2 we have to add it to_be_onehotencoded_cols list
to_be_onehotencoded_cols = []

for col_name in categoric_cols:
    count = df.select(col_name).distinct().count()
    if count > 2:
        to_be_onehotencoded_cols.append(col_name)
    print("{} has {} distinct category.".format(col_name, count))
    

Geography has 3 distinct category.
Gender has 2 distinct category.


In [34]:
print(to_be_onehotencoded_cols)

['Geography']


# StringIndexer

In [35]:
from pyspark.ml.feature import StringIndexer

In [36]:
# Will hold stringIndexer objects and column names
my_dict = {}

# Will collect StringIndexer ojects
string_indexer_objs = []

# Will collect StringIndexer output colnames
string_indexer_output_names = []

# Will collect OneHotEncoder output colnames
ohe_col_input_names = []
ohe_col_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_indexobj"] = StringIndexer() \
    .setHandleInvalid('skip') \
    .setInputCol(col_name) \
    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_indexobj"))
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in to_be_onehotencoded_cols:
        ohe_col_input_names.append(col_name+"_indexed")
        ohe_col_output_names.append(col_name+"_ohe")

In [37]:
print(string_indexer_objs)
print(len(string_indexer_objs))

[StringIndexer_817798891130, StringIndexer_657950a2fbc3]
2


In [38]:
print(string_indexer_output_names)
print(len(string_indexer_output_names))

['Geography_indexed', 'Gender_indexed']
2


In [39]:
print(ohe_col_input_names)
print(len(ohe_col_input_names))

['Geography_indexed']
1


In [40]:
print(ohe_col_output_names)
print(len(ohe_col_output_names))

['Geography_ohe']
1


# OnehotEncoder

In [41]:
from pyspark.ml.feature import OneHotEncoder

In [42]:
encoder = OneHotEncoder() \
.setInputCols(ohe_col_input_names) \
.setOutputCols(ohe_col_output_names)

# VectorAssembler

In [43]:
from pyspark.ml.feature import VectorAssembler

In [44]:
# We have to exclude onehotencoded categoric cols from stringIndexer cols
string_indexer_col_names_ohe_exluded = list(set(string_indexer_output_names).difference(set(ohe_col_input_names)))
string_indexer_col_names_ohe_exluded

['Gender_indexed']

## column verification

In [45]:
if len(categoric_cols) == (len(string_indexer_col_names_ohe_exluded)+len(ohe_col_output_names)):
    print("Columns verified")
else: print("Columns are not verified")

Columns verified


In [46]:
assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+string_indexer_col_names_ohe_exluded+ohe_col_output_names) \
.setOutputCol('unscaled_features')

# LabelIndexer

In [47]:
label_indexer = StringIndexer().setHandleInvalid("skip") \
.setInputCol(label_col[0]) \
.setOutputCol("label")

# Feature Scaling

In [48]:
from pyspark.ml.feature import StandardScaler

In [49]:
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("features")

# PCA

In [69]:
# from pyspark.ml.feature import PCA

In [70]:
# pca = PCA().setInputCol("scaled_features").setK(10).setOutputCol("features")

# Estimator: GBTClassifier

In [51]:
from pyspark.ml.classification import GBTClassifier

In [52]:
estimator = GBTClassifier() \
.setFeaturesCol("features") \
.setLabelCol(label_col[0])

# Pipeline

In [53]:
from pyspark.ml import Pipeline

In [54]:
pipeline_obj = Pipeline().setStages(string_indexer_objs+[encoder, assembler, label_indexer, scaler, estimator])

In [62]:
pipeline_prep_obj = Pipeline().setStages(string_indexer_objs+[encoder, assembler, label_indexer, scaler])

In [63]:
pipeline_prep_model = pipeline_prep_obj.fit(train_df)

In [64]:
pipeline_prep_model.transform(test_df).limit(5).toPandas()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_indexed,Gender_indexed,Geography_ohe,unscaled_features,label,features
0,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.0,1.0,"(1.0, 0.0)","[699.0, 39.0, 1.0, 0.0, 2.0, 0.0, 0.0, 93826.6...",0.0,"[7.249531660399154, 3.720116711013842, 0.34679..."
1,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0,0.0,0.0,"(1.0, 0.0)","[501.0, 44.0, 4.0, 142051.07, 2.0, 0.0, 1.0, 7...",0.0,"[5.196016254449179, 4.197054750887411, 1.38719..."
2,11,15767821,Bearce,528,France,Male,31,6,102016.72,2,0,0,80181.12,0,0.0,0.0,"(1.0, 0.0)","[528.0, 31.0, 6.0, 102016.72, 2.0, 0.0, 0.0, 8...",0.0,"[5.476041082533267, 2.9570158472161308, 2.0807..."
3,15,15600882,Scott,635,Spain,Female,35,7,0.00,2,1,1,65951.65,0,2.0,1.0,"(0.0, 0.0)","[635.0, 35.0, 7.0, 0.0, 2.0, 1.0, 1.0, 65951.6...",0.0,"[6.585769104940576, 3.3385662791149864, 2.4275..."
4,25,15625047,Yen,846,France,Female,38,5,0.00,1,1,1,187616.16,0,0.0,1.0,"(1.0, 0.0)","[846.0, 38.0, 5.0, 0.0, 1.0, 1.0, 1.0, 187616....",0.0,"[8.774111279968075, 3.6247291030391278, 1.7339..."


# Split data

In [55]:
train_df, test_df = df.randomSplit([0.8,0.2], seed=142)

In [56]:
print(train_df.count())

7994


# Train Model

In [57]:
pipeline_model = pipeline_obj.fit(train_df)

In [58]:
pipeline_model.stages

[StringIndexerModel: uid=StringIndexer_817798891130, handleInvalid=skip,
 StringIndexerModel: uid=StringIndexer_657950a2fbc3, handleInvalid=skip,
 OneHotEncoderModel: uid=OneHotEncoder_44abba697904, dropLast=true, handleInvalid=error, numInputCols=1, numOutputCols=1,
 VectorAssembler_dabf78e8b04a,
 StringIndexerModel: uid=StringIndexer_e55cee05d3b6, handleInvalid=skip,
 StandardScalerModel: uid=StandardScaler_1945294108aa, numFeatures=11, withMean=false, withStd=true,
 GBTClassificationModel: uid = GBTClassifier_b4be44504932, numTrees=20, numClasses=2, numFeatures=11]

# Save the model

In [59]:
pipeline_model.write().overwrite().save("file:///home/train/atscale5/pyspark/08_ml/saved_models/gbt_churn_10K")

# Load Saved Model

In [60]:
from pyspark.ml.pipeline import PipelineModel

In [61]:
pipeline_model_loaded = PipelineModel.load("file:///home/train/atscale5/pyspark/08_ml/saved_models/gbt_churn_10K")

# Prediction

In [65]:
transformed_df = pipeline_model_loaded.transform(test_df)

In [66]:
transformed_df.limit(5).toPandas()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_indexed,Gender_indexed,Geography_ohe,unscaled_features,label,features,rawPrediction,probability,prediction
0,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.0,1.0,"(1.0, 0.0)","[699.0, 39.0, 1.0, 0.0, 2.0, 0.0, 0.0, 93826.6...",0.0,"[7.249531660399154, 3.720116711013842, 0.34679...","[1.2158222417091507, -1.2158222417091507]","[0.9192087429720839, 0.08079125702791612]",0.0
1,9,15792365,He,501,France,Male,44,4,142051.07,2,0,1,74940.50,0,0.0,0.0,"(1.0, 0.0)","[501.0, 44.0, 4.0, 142051.07, 2.0, 0.0, 1.0, 7...",0.0,"[5.196016254449179, 4.197054750887411, 1.38719...","[1.2223934623292743, -1.2223934623292743]","[0.9201793919301929, 0.07982060806980706]",0.0
2,11,15767821,Bearce,528,France,Male,31,6,102016.72,2,0,0,80181.12,0,0.0,0.0,"(1.0, 0.0)","[528.0, 31.0, 6.0, 102016.72, 2.0, 0.0, 0.0, 8...",0.0,"[5.476041082533267, 2.9570158472161308, 2.0807...","[1.2975955516181858, -1.2975955516181858]","[0.9305514452488673, 0.06944855475113265]",0.0
3,15,15600882,Scott,635,Spain,Female,35,7,0.00,2,1,1,65951.65,0,2.0,1.0,"(0.0, 0.0)","[635.0, 35.0, 7.0, 0.0, 2.0, 1.0, 1.0, 65951.6...",0.0,"[6.585769104940576, 3.3385662791149864, 2.4275...","[1.3508182501964772, -1.3508182501964772]","[0.9371231410519548, 0.06287685894804518]",0.0
4,25,15625047,Yen,846,France,Female,38,5,0.00,1,1,1,187616.16,0,0.0,1.0,"(1.0, 0.0)","[846.0, 38.0, 5.0, 0.0, 1.0, 1.0, 1.0, 187616....",0.0,"[8.774111279968075, 3.6247291030391278, 1.7339...","[0.8030550367358321, -0.8030550367358321]","[0.8328706206266746, 0.1671293793733254]",0.0


In [67]:
transformed_df.select('label','prediction','probability','rawPrediction').show(truncate=False)

+-----+----------+-----------------------------------------+--------------------------------------------+
|label|prediction|probability                              |rawPrediction                               |
+-----+----------+-----------------------------------------+--------------------------------------------+
|0.0  |0.0       |[0.9192087429720839,0.08079125702791612] |[1.2158222417091507,-1.2158222417091507]    |
|0.0  |0.0       |[0.9201793919301929,0.07982060806980706] |[1.2223934623292743,-1.2223934623292743]    |
|0.0  |0.0       |[0.9305514452488673,0.06944855475113265] |[1.2975955516181858,-1.2975955516181858]    |
|0.0  |0.0       |[0.9371231410519548,0.06287685894804518] |[1.3508182501964772,-1.3508182501964772]    |
|0.0  |0.0       |[0.8328706206266746,0.1671293793733254]  |[0.8030550367358321,-0.8030550367358321]    |
|0.0  |1.0       |[0.487459196625952,0.512540803374048]    |[-0.025086868238423474,0.025086868238423474]|
|0.0  |0.0       |[0.9143698738559843,0.085630

# Evaluate the model

In [70]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [69]:
evaluator = BinaryClassificationEvaluator()

In [64]:
evaluator.evaluate(transformed_df)

0.8591799680023813

In [65]:
evaluator.getMetricName()

'areaUnderROC'

In [71]:
mc_evaluator = MulticlassClassificationEvaluator(metricName='accuracy')

In [72]:
mc_evaluator.evaluate(transformed_df)

0.8529411764705882

In [73]:
transformed_df.select('label','prediction','probability','rawPrediction') \
.filter("prediction == 1.0") \
.show(truncate=False)

+-----+----------+-----------------------------------------+--------------------------------------------+
|label|prediction|probability                              |rawPrediction                               |
+-----+----------+-----------------------------------------+--------------------------------------------+
|0.0  |1.0       |[0.487459196625952,0.512540803374048]    |[-0.025086868238423474,0.025086868238423474]|
|1.0  |1.0       |[0.48285217871390007,0.5171478212860999] |[-0.034309098145787024,0.034309098145787024]|
|1.0  |1.0       |[0.0956753908494807,0.9043246091505193]  |[-1.1231136301872786,1.1231136301872786]    |
|1.0  |1.0       |[0.2911612744236413,0.7088387255763586]  |[-0.4448753559543522,0.4448753559543522]    |
|1.0  |1.0       |[0.1064385517352367,0.8935614482647634]  |[-1.0638236325769461,1.0638236325769461]    |
|1.0  |1.0       |[0.44401831433515854,0.5559816856648414] |[-0.11243477208610242,0.11243477208610242]  |
|1.0  |1.0       |[0.033737796593392745,0.9662

In [74]:
transformed_df.select('label','prediction','probability','rawPrediction') \
.withColumn("score", F.col('probability').cast('string')).show()

+-----+----------+--------------------+--------------------+--------------------+
|label|prediction|         probability|       rawPrediction|               score|
+-----+----------+--------------------+--------------------+--------------------+
|  0.0|       0.0|[0.91920874297208...|[1.21582224170915...|[0.91920874297208...|
|  0.0|       0.0|[0.92017939193019...|[1.22239346232927...|[0.92017939193019...|
|  0.0|       0.0|[0.93055144524886...|[1.29759555161818...|[0.93055144524886...|
|  0.0|       0.0|[0.93712314105195...|[1.35081825019647...|[0.93712314105195...|
|  0.0|       0.0|[0.83287062062667...|[0.80305503673583...|[0.83287062062667...|
|  0.0|       1.0|[0.48745919662595...|[-0.0250868682384...|[0.48745919662595...|
|  0.0|       0.0|[0.91436987385598...|[1.18409900171018...|[0.91436987385598...|
|  0.0|       0.0|[0.66738020252566...|[0.34817990696696...|[0.66738020252566...|
|  1.0|       0.0|[0.83358038098355...|[0.80560885562677...|[0.83358038098355...|
|  0.0|       0.

In [75]:
transformed_df.select('label','prediction','probability','rawPrediction').printSchema()

root
 |-- label: double (nullable = false)
 |-- prediction: double (nullable = false)
 |-- probability: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)



# Get churn score from probability

In [76]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

to_array = F.udf(lambda v: v.toArray().tolist(), T.ArrayType(T.FloatType()))
score_df = transformed_df.select('CustomerID','label','prediction','probability','rawPrediction') \
.withColumn('score', to_array('probability')[1] * 1000)

In [77]:
score_df.filter("prediction == 1.0").orderBy(F.desc("score")).show()

+----------+-----+----------+--------------------+--------------------+---------+
|CustomerID|label|prediction|         probability|       rawPrediction|    score|
+----------+-----+----------+--------------------+--------------------+---------+
|  15691817|  1.0|       1.0|[0.02369262962039...|[-1.8593067261036...| 976.3074|
|  15664754|  1.0|       1.0|[0.02846758090187...|[-1.7650543554032...| 971.5324|
|  15621550|  1.0|       1.0|[0.03311205302199...|[-1.6870926280199...|966.88794|
|  15764866|  1.0|       1.0|[0.03373779659339...|[-1.6774082299827...| 966.2622|
|  15606386|  1.0|       1.0|[0.03693969119183...|[-1.6304147103965...| 963.0603|
|  15568006|  1.0|       1.0|[0.03711859300247...|[-1.6279061257538...| 962.8814|
|  15597914|  1.0|       1.0|[0.03828019402119...|[-1.6118952556091...| 961.7198|
|  15633194|  1.0|       1.0|[0.03899978276177...|[-1.6022092795607...| 961.0002|
|  15641675|  1.0|       1.0|[0.03941582980987...|[-1.5966870512847...|960.58417|
|  15605826|  1.

# Top 20 Costumers most likely to churn

In [78]:
score_df.orderBy(F.desc("score")).show()

+----------+-----+----------+--------------------+--------------------+---------+
|CustomerID|label|prediction|         probability|       rawPrediction|    score|
+----------+-----+----------+--------------------+--------------------+---------+
|  15691817|  1.0|       1.0|[0.02369262962039...|[-1.8593067261036...| 976.3074|
|  15664754|  1.0|       1.0|[0.02846758090187...|[-1.7650543554032...| 971.5324|
|  15621550|  1.0|       1.0|[0.03311205302199...|[-1.6870926280199...|966.88794|
|  15764866|  1.0|       1.0|[0.03373779659339...|[-1.6774082299827...| 966.2622|
|  15606386|  1.0|       1.0|[0.03693969119183...|[-1.6304147103965...| 963.0603|
|  15568006|  1.0|       1.0|[0.03711859300247...|[-1.6279061257538...| 962.8814|
|  15597914|  1.0|       1.0|[0.03828019402119...|[-1.6118952556091...| 961.7198|
|  15633194|  1.0|       1.0|[0.03899978276177...|[-1.6022092795607...| 961.0002|
|  15641675|  1.0|       1.0|[0.03941582980987...|[-1.5966870512847...|960.58417|
|  15605826|  1.

In [79]:
score_df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- label: double (nullable = false)
 |-- prediction: double (nullable = false)
 |-- probability: vector (nullable = true)
 |-- rawPrediction: vector (nullable = true)
 |-- score: float (nullable = true)



# Hyperparameter Tuning

In [81]:
from pyspark.ml.tuning import CrossValidatorModel, ParamGridBuilder, CrossValidator

In [82]:
paramGrid = ParamGridBuilder() \
.addGrid(estimator.maxDepth, [5, 10]) \
.addGrid(estimator.maxBins, [32, 50]) \
.build()

In [83]:
crossval = CrossValidator(estimator=pipeline_obj,
                          estimatorParamMaps=paramGrid,
                          evaluator=BinaryClassificationEvaluator(),
                          numFolds=5) 

In [111]:
cvModel = crossval.fit(train_df)

In [112]:
type(cvModel)

pyspark.ml.tuning.CrossValidatorModel

In [113]:
cvModel.params

[Param(parent='CrossValidatorModel_da2736282f15', name='estimator', doc='estimator to be cross-validated'),
 Param(parent='CrossValidatorModel_da2736282f15', name='estimatorParamMaps', doc='estimator param maps'),
 Param(parent='CrossValidatorModel_da2736282f15', name='evaluator', doc='evaluator used to select hyper-parameters that maximize the validator metric'),
 Param(parent='CrossValidatorModel_da2736282f15', name='numFolds', doc='number of folds for cross validation'),
 Param(parent='CrossValidatorModel_da2736282f15', name='seed', doc='random seed.')]

In [114]:
bestModel = cvModel.bestModel

In [115]:
evaluator.evaluate(bestModel.transform(test_df))

0.8591799680023813

In [116]:
evaluator.evaluate(cvModel.transform(test_df))

0.8591799680023813

In [ ]:
spark.stop()